# Notebook 10 — Full Pipeline Integration

**Phase 4 · Evaluation & Deployment (2 / 2)**

---

## 🎯 Learning Objectives

| # | Goal |
|---|------|
| 1 | Wire every module into a single **end-to-end trading loop** |
| 2 | Understand the production `TradingBot` lifecycle: bootstrap → poll → strategy cycle |
| 3 | Simulate a multi-day loop: data → signals → regime → ensemble → portfolio → risk → execution |
| 4 | Implement **state persistence** (save/load JSON) |
| 5 | Review the **production architecture** and compare with our walkthrough |

### Prerequisites
- NB01–NB09 (all prior notebooks)

In [ ]:
# ── Setup ──────────────────────────────────────────────────
import sys, pathlib, warnings, json, math
from datetime import datetime, timezone
from dataclasses import dataclass, field

warnings.filterwarnings("ignore")
ROOT = str(pathlib.Path.cwd().resolve().parents[1])
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams.update({"figure.figsize": (14, 5), "axes.grid": True})
print("✅ Imports OK  |  Project root:", ROOT)

---
## 1 · Production Architecture Overview

The production `TradingBot` in `main.py` orchestrates six layers:

```
┌─────────────────────────────────────────────────────────────┐
│                      TradingBot                            │
│                                                             │
│  ┌──────────┐  ┌──────────┐  ┌──────────┐  ┌────────────┐ │
│  │   Data   │→│  Signals  │→│ Strategy  │→│  Portfolio  │ │
│  │ Binance  │  │ Momentum │  │ Regime    │  │ Normalize  │ │
│  │ OHLCV    │  │ MeanRev  │  │ Ensemble  │  │ Optimize   │ │
│  │ Ticker   │  │ Pairs    │  │ Sentiment │  │            │ │
│  └──────────┘  │ Sector   │  └──────────┘  └────────────┘ │
│                └──────────┘        │              │        │
│                                    ▼              ▼        │
│  ┌────────────┐  ┌───────────────────────────────────────┐ │
│  │ Monitoring  │  │           Risk & Execution            │ │
│  │ Telegram    │  │  Circuit Breakers → Position Limits   │ │
│  │ Metrics     │  │  → Order Generation → API Submit      │ │
│  └────────────┘  └───────────────────────────────────────┘ │
│                                                             │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  Scheduler: poll (60s) → strategy (300s) → heartbeat│   │
│  └─────────────────────────────────────────────────────┘   │
└─────────────────────────────────────────────────────────────┘
```

### Key Lifecycle Events

| Event | Method | Frequency |
|-------|--------|-----------|
| **Bootstrap** | `bootstrap()` | Once at startup |
| **Ticker Poll** | `run_poll_cycle()` | Every 60 seconds |
| **Strategy Cycle** | `run_operational_cycle()` | Every 300 seconds |
| **Heartbeat** | `send_heartbeat()` | Every 3600 seconds |
| **Clock Sync** | `sync_server_time()` | Every 3600 seconds |

---
## 2 · Synthetic Data Generation

In [ ]:
np.random.seed(42)
N_DAYS = 120
dates = pd.date_range(end=pd.Timestamp.now().normalize(), periods=N_DAYS, freq="D")

assets = ["BTCUSDT", "ETHUSDT", "SOLUSDT", "BNBUSDT", "XRPUSDT",
          "ADAUSDT", "AVAXUSDT", "DOTUSDT"]

# Generate correlated returns with regime structure
market = np.random.normal(0.001, 0.015, N_DAYS)
# Inject a bear regime in the middle
market[40:60] -= 0.012
# Inject a bull regime later
market[80:100] += 0.008

prices = pd.DataFrame(index=dates)
base_prices = {"BTCUSDT": 60000, "ETHUSDT": 3500, "SOLUSDT": 150, "BNBUSDT": 600,
               "XRPUSDT": 0.60, "ADAUSDT": 0.45, "AVAXUSDT": 35, "DOTUSDT": 7}

for i, asset in enumerate(assets):
    beta = 0.6 + 0.4 * i / len(assets)
    alpha = 0.0003 * (len(assets) - i) / len(assets)
    idio = np.random.normal(0, 0.012, N_DAYS)
    r = alpha + beta * market + idio
    prices[asset] = base_prices[asset] * np.exp(np.cumsum(r))

# Volume (synthetic)
volumes = pd.DataFrame(
    np.random.lognormal(mean=15, sigma=1.5, size=(N_DAYS, len(assets))),
    index=dates, columns=assets
)

print(f"Price panel: {prices.shape[0]} days × {prices.shape[1]} assets")
print(f"Regime structure: bear @ days 40-60, bull @ days 80-100")
prices.tail(3)

---
## 3 · Building Each Pipeline Stage

We re-implement each module as a standalone function, then chain them together.

In [ ]:
# ── Stage 1: Regime Detection ─────────────────────────────

def detect_regime(prices_btc: pd.Series, ema_short: int = 20, ema_long: int = 50,
                  vol_lookback: int = 14, vol_multiplier: float = 1.5) -> str:
    """Classify current market regime as bull, bear, or ranging."""
    if len(prices_btc) < ema_long + 1:
        return "ranging"
    ema_s = prices_btc.ewm(span=ema_short, adjust=False).mean()
    ema_l = prices_btc.ewm(span=ema_long, adjust=False).mean()
    returns = prices_btc.pct_change().dropna()
    recent_vol = returns.iloc[-vol_lookback:].std()
    baseline_vol = returns.std()
    trending = ema_s.iloc[-1] > ema_l.iloc[-1]
    high_vol = recent_vol > baseline_vol * vol_multiplier
    if trending and not high_vol:
        return "bull"
    elif not trending and high_vol:
        return "bear"
    return "ranging"

# Test
regime = detect_regime(prices["BTCUSDT"])
print(f"Current regime: {regime}")

In [ ]:
# ── Stage 2: Signal Generation ────────────────────────────

def momentum_signal(prices: pd.DataFrame, lookbacks: list = [3, 5, 7],
                    top_n: int = 5) -> dict[str, float]:
    """Rank assets by multi-lookback momentum, return weight map."""
    scores = pd.Series(0.0, index=prices.columns)
    for lb in lookbacks:
        ret = prices.pct_change(lb).iloc[-1]
        norm = (ret - ret.min()) / (ret.max() - ret.min() + 1e-10)
        scores += norm
    scores /= len(lookbacks)
    top = scores.nlargest(top_n)
    total = top.sum()
    if total == 0:
        return {sym: 1.0 / top_n for sym in top.index}
    return {sym: float(w / total) for sym, w in top.items()}

def mean_reversion_signal(prices: pd.DataFrame, rsi_period: int = 14,
                          rsi_oversold: float = 30) -> dict[str, float]:
    """Find oversold assets by RSI, return weight map."""
    weights = {}
    for asset in prices.columns:
        delta = prices[asset].diff()
        gain = delta.clip(lower=0).rolling(rsi_period).mean()
        loss = (-delta.clip(upper=0)).rolling(rsi_period).mean()
        rs = gain / (loss + 1e-10)
        rsi = 100 - (100 / (1 + rs))
        current_rsi = rsi.iloc[-1]
        if current_rsi < rsi_oversold:
            weights[asset] = float(1.0 - current_rsi / 100.0)
    if weights:
        total = sum(weights.values())
        return {k: v / total for k, v in weights.items()}
    return {}

# Generate signals
mom_weights = momentum_signal(prices)
mr_weights = mean_reversion_signal(prices)
print(f"Momentum picks {len(mom_weights)} assets: {list(mom_weights.keys())}")
print(f"Mean-reversion picks {len(mr_weights)} assets: {list(mr_weights.keys())}")

In [ ]:
# ── Stage 3: Ensemble Blending ────────────────────────────

REGIME_WEIGHTS = {
    "bull":    {"momentum": 0.50, "mean_reversion": 0.10, "sentiment": 0.20, "sector": 0.20},
    "ranging": {"momentum": 0.20, "mean_reversion": 0.50, "sentiment": 0.30, "sector": 0.00},
    "bear":    {"momentum": 0.00, "mean_reversion": 0.30, "sentiment": 0.20, "sector": 0.00},
}

CASH_FLOORS = {"bull": 0.20, "ranging": 0.40, "bear": 0.50}

def ensemble_combine(regime: str, signal_maps: dict[str, dict]) -> dict[str, float]:
    """Blend sub-strategy weight maps using regime-dependent multipliers."""
    blend = REGIME_WEIGHTS.get(regime, REGIME_WEIGHTS["ranging"])
    combined = {}
    for strategy_name, weight_map in signal_maps.items():
        factor = blend.get(strategy_name, 0.0)
        for sym, w in weight_map.items():
            combined[sym] = combined.get(sym, 0.0) + w * factor
    return combined

raw_ensemble = ensemble_combine(regime, {
    "momentum": mom_weights,
    "mean_reversion": mr_weights,
})
print(f"Regime: {regime}")
print(f"Raw ensemble weights ({len(raw_ensemble)} assets):")
for sym, w in sorted(raw_ensemble.items(), key=lambda x: -x[1]):
    print(f"  {sym:12s} {w:.4f}")

In [ ]:
# ── Stage 4: Portfolio Normalization ──────────────────────

def normalize_weights(weights: dict[str, float], cash_floor: float) -> dict[str, float]:
    """Scale positive weights to sum to (1 - cash_floor)."""
    positives = {k: v for k, v in weights.items() if v > 0}
    if not positives:
        return {}
    total = sum(positives.values())
    target_sum = 1.0 - cash_floor
    return {k: v / total * target_sum for k, v in positives.items()}

cash_floor = CASH_FLOORS[regime]
normalized = normalize_weights(raw_ensemble, cash_floor)

print(f"Cash floor ({regime}): {cash_floor:.0%}")
print(f"Invested:  {sum(normalized.values()):.2%}")
print(f"Cash:      {1 - sum(normalized.values()):.2%}")
for sym, w in sorted(normalized.items(), key=lambda x: -x[1]):
    print(f"  {sym:12s} {w:.4f}")

In [ ]:
# ── Stage 5: Risk Controls ────────────────────────────────

def enforce_position_limit(weights: dict[str, float], max_position: float = 0.10) -> dict[str, float]:
    """Clip each position to max_position."""
    return {k: min(v, max_position) for k, v in weights.items()}

def check_circuit_breaker(drawdown: float, l1: float = 0.03, l2: float = 0.05) -> str:
    """Return 'halt', 'reduce', or 'ok'."""
    if abs(drawdown) >= l2:
        return "halt"
    elif abs(drawdown) >= l1:
        return "reduce"
    return "ok"

risk_adjusted = enforce_position_limit(normalized, max_position=0.10)
print(f"After position limits (max 10%):")
for sym, w in sorted(risk_adjusted.items(), key=lambda x: -x[1]):
    print(f"  {sym:12s} {w:.4f}")

In [ ]:
# ── Stage 6: Order Generation ─────────────────────────────

def generate_rebalance_orders(current_weights: dict[str, float],
                               target_weights: dict[str, float],
                               min_drift: float = 0.15) -> list[dict]:
    """Generate BUY/SELL orders for weights that drifted beyond threshold."""
    all_symbols = set(list(current_weights.keys()) + list(target_weights.keys()))
    orders = []
    for sym in sorted(all_symbols):
        curr = current_weights.get(sym, 0.0)
        tgt = target_weights.get(sym, 0.0)
        drift = abs(tgt - curr) / max(curr, 1e-10)
        if drift > min_drift:
            side = "BUY" if tgt > curr else "SELL"
            orders.append({"side": side, "symbol": sym, "target_weight": tgt, "drift": drift})
    return orders

# Start from no positions
current_positions = {asset: 0.0 for asset in assets}
orders = generate_rebalance_orders(current_positions, risk_adjusted)

print(f"Generated {len(orders)} orders:")
for o in orders:
    print(f"  {o['side']:4s} {o['symbol']:12s} target={o['target_weight']:.4f}")

---
## 4 · State Persistence

The production bot saves state to `data/bot_state.json` using **atomic writes** (write to temp file, then rename). Here's a simplified version:

In [ ]:
import tempfile, os

def save_state(state: dict, filepath: str = "/tmp/bot_state_demo.json") -> None:
    """Atomic save: write to temp, then rename."""
    directory = os.path.dirname(filepath)
    fd, tmp_path = tempfile.mkstemp(dir=directory, suffix=".tmp")
    try:
        with os.fdopen(fd, "w") as f:
            json.dump(state, f, indent=2, sort_keys=True, default=str)
            f.write("\n")
        os.replace(tmp_path, filepath)
    except:
        if os.path.exists(tmp_path):
            os.unlink(tmp_path)
        raise

def load_state(filepath: str = "/tmp/bot_state_demo.json") -> dict:
    """Load state from JSON file."""
    if not os.path.exists(filepath):
        return {}
    with open(filepath, "r") as f:
        return json.load(f)

# Demo state
demo_state = {
    "portfolio_value": 10000.0,
    "positions": current_positions,
    "regime": regime,
    "last_updated": datetime.now(timezone.utc).isoformat(),
}
save_state(demo_state)
loaded = load_state()
print(f"Saved and loaded state: {list(loaded.keys())}")
print(f"Portfolio value: ${loaded['portfolio_value']:,.2f}")

---
## 5 · Full Pipeline Simulation (Multi-Day Loop)

Now we chain all stages into a **daily trading loop** that processes 120 days of data:

In [ ]:
# ── Full Pipeline Configuration ──────────────────────────
CONFIG = {
    "regime": {"ema_short": 20, "ema_long": 50, "vol_lookback": 14, "vol_multiplier": 1.5},
    "momentum": {"lookbacks": [3, 5, 7], "top_n": 5},
    "mean_reversion": {"rsi_period": 14, "rsi_oversold": 30},
    "risk": {"max_position": 0.10, "circuit_breaker_l1": 0.03, "circuit_breaker_l2": 0.05},
    "execution": {"min_drift": 0.15},
}

# ── Pipeline Loop ────────────────────────────────────────
WARMUP = max(CONFIG["regime"]["ema_long"], CONFIG["mean_reversion"]["rsi_period"]) + 5

portfolio_value = 10000.0
current_weights = {asset: 0.0 for asset in assets}
equity_curve = []
regime_history = []
order_log = []
circuit_breaker_log = []

peak_value = portfolio_value

for day in range(WARMUP, N_DAYS):
    date = dates[day]
    window = prices.iloc[:day + 1]
    
    # Stage 1: Regime
    reg = detect_regime(window["BTCUSDT"], **CONFIG["regime"])
    regime_history.append({"date": date, "regime": reg})
    
    # Stage 2: Signals
    mom_w = momentum_signal(window, **CONFIG["momentum"])
    mr_w = mean_reversion_signal(window, **CONFIG["mean_reversion"])
    
    # Stage 3: Ensemble
    raw = ensemble_combine(reg, {"momentum": mom_w, "mean_reversion": mr_w})
    
    # Stage 4: Normalize
    cf = CASH_FLOORS[reg]
    norm_w = normalize_weights(raw, cf)
    
    # Stage 5: Risk
    drawdown = (portfolio_value / peak_value) - 1 if peak_value > 0 else 0
    cb_status = check_circuit_breaker(drawdown, **{k.replace("circuit_breaker_", ""): v 
                                                    for k, v in CONFIG["risk"].items() 
                                                    if "circuit" in k})
    circuit_breaker_log.append({"date": date, "status": cb_status, "drawdown": drawdown})
    
    if cb_status == "halt":
        target_w = {asset: 0.0 for asset in assets}
    elif cb_status == "reduce":
        target_w = {k: v * 0.5 for k, v in norm_w.items()}
    else:
        target_w = enforce_position_limit(norm_w, CONFIG["risk"]["max_position"])
    
    # Stage 6: Orders
    day_orders = generate_rebalance_orders(current_weights, target_w, CONFIG["execution"]["min_drift"])
    if day_orders:
        order_log.append({"date": date, "count": len(day_orders), "regime": reg})
    
    # Apply weights (simulate)
    current_weights = target_w
    
    # Compute portfolio return for this day
    if day < N_DAYS - 1:
        daily_ret = prices.iloc[day + 1] / prices.iloc[day] - 1
        port_return = sum(current_weights.get(a, 0) * daily_ret.get(a, 0) for a in assets)
        portfolio_value *= (1 + port_return)
        peak_value = max(peak_value, portfolio_value)
    
    equity_curve.append({"date": date, "value": portfolio_value, "regime": reg})

eq_df = pd.DataFrame(equity_curve).set_index("date")
print(f"Simulation complete: {len(equity_curve)} days")
print(f"Final portfolio: ${portfolio_value:,.2f}  ({(portfolio_value/10000 - 1)*100:+.2f}%)")
print(f"Rebalance events: {len(order_log)}")

---
## 6 · Pipeline Visualization

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True,
                          gridspec_kw={"height_ratios": [3, 1, 1]})

# ── Panel 1: Equity Curve with Regime Background ─────────
ax1 = axes[0]
ax1.plot(eq_df.index, eq_df["value"], lw=1.5, color="#2c3e50", label="Portfolio")

# Buy & hold benchmark
bench_ret = prices.iloc[WARMUP:].pct_change().mean(axis=1)
bench_eq = 10000 * (1 + bench_ret).cumprod()
ax1.plot(bench_eq.index, bench_eq.values, lw=1.2, color="#95a5a6", ls="--", label="Equal-Weight B&H")

# Regime shading
colors = {"bull": "#2ecc71", "ranging": "#f1c40f", "bear": "#e74c3c"}
for i in range(len(eq_df) - 1):
    ax1.axvspan(eq_df.index[i], eq_df.index[i+1], alpha=0.08,
                color=colors.get(eq_df["regime"].iloc[i], "gray"))

ax1.set_ylabel("Portfolio Value ($)")
ax1.set_title("Full Pipeline Simulation")
patches = [mpatches.Patch(color=c, alpha=0.3, label=r) for r, c in colors.items()]
ax1.legend(handles=[*ax1.get_legend_handles_labels()[0][:2], *patches], loc="upper left", fontsize=8)

# ── Panel 2: Drawdown ────────────────────────────────────
ax2 = axes[1]
equity_series = eq_df["value"]
peak = equity_series.cummax()
dd = (equity_series / peak) - 1
ax2.fill_between(dd.index, dd, alpha=0.4, color="#e74c3c")
ax2.axhline(-0.03, ls="--", color="orange", lw=0.8, label="L1 (-3%)")
ax2.axhline(-0.05, ls="--", color="red", lw=0.8, label="L2 (-5%)")
ax2.set_ylabel("Drawdown")
ax2.legend(fontsize=8)
ax2.invert_yaxis()

# ── Panel 3: Rebalance Activity ──────────────────────────
ax3 = axes[2]
if order_log:
    ol_df = pd.DataFrame(order_log)
    bar_colors = [colors.get(r, "gray") for r in ol_df["regime"]]
    ax3.bar(ol_df["date"], ol_df["count"], color=bar_colors, alpha=0.7, width=1)
ax3.set_ylabel("# Orders")
ax3.set_xlabel("Date")

plt.tight_layout()
plt.show()

---
## 7 · Pipeline Statistics

In [ ]:
regime_counts = pd.DataFrame(regime_history)["regime"].value_counts()
cb_df = pd.DataFrame(circuit_breaker_log)
cb_counts = cb_df["status"].value_counts()

print("═" * 50)
print(" PIPELINE SUMMARY")
print("═" * 50)
print(f"\n  Simulation period:  {eq_df.index[0].date()} → {eq_df.index[-1].date()}")
print(f"  Trading days:       {len(equity_curve)}")
print(f"  Starting capital:   $10,000.00")
print(f"  Final value:        ${portfolio_value:,.2f}")
print(f"  Total return:       {(portfolio_value/10000 - 1)*100:+.2f}%")
print(f"  Max drawdown:       {dd.min():.2%}")

print(f"\n  Regime Distribution:")
for r, count in regime_counts.items():
    print(f"    {r:10s} {count:4d} days ({count/len(regime_history)*100:.1f}%)")

print(f"\n  Circuit Breaker Triggers:")
for status, count in cb_counts.items():
    print(f"    {status:10s} {count:4d} days")

print(f"\n  Rebalance events:   {len(order_log)}")
if order_log:
    total_orders = sum(o["count"] for o in order_log)
    print(f"  Total orders:       {total_orders}")

---
## 8 · Production Comparison

How our simulation maps to the production `TradingBot` class:

| Our Simulation | Production Code | File |
|----------------|----------------|------|
| `detect_regime()` | `detect_regime()` | `bot/strategy/regime_detector.py` |
| `momentum_signal()` | `rank_assets_by_momentum()` | `bot/signals/momentum.py` |
| `mean_reversion_signal()` | `find_oversold_assets()` | `bot/signals/mean_reversion.py` |
| `ensemble_combine()` | `ensemble_combine()` | `bot/strategy/ensemble.py` |
| `normalize_weights()` | `normalize_weights()` | `bot/strategy/portfolio_optimizer.py` |
| `enforce_position_limit()` | `enforce_position_limit()` | `bot/risk/risk_manager.py` |
| `check_circuit_breaker()` | `CircuitBreaker.evaluate()` | `bot/risk/circuit_breaker.py` |
| `generate_rebalance_orders()` | `generate_rebalance_orders()` | `bot/execution/order_executor.py` |
| `save_state()` / `load_state()` | `TradingBot.save_state()` / `load_state()` | `bot/main.py` |
| Day loop | `APScheduler` (60s poll / 300s strategy) | `bot/main.py` |

### Production Additions

| Feature | Description |
|---------|-------------|
| **Roostoo API** | Real order execution via REST API |
| **Telegram Alerts** | Heartbeat + trade notifications |
| **Clock Sync** | Server time offset for accurate timestamps |
| **Atomic State Writes** | `tempfile.mkstemp()` + `os.replace()` |
| **APScheduler** | Cron-like job scheduling |
| **Pairs Rotation** | Cointegration-based pairs trading |
| **Sector Rotation** | BTC dominance based allocation |
| **Sentiment Overlay** | Fear & Greed Index multiplier |

In [ ]:
# Quick check: import production classes to verify accessibility
from bot.main import TradingBot, StrategyCycleResult
from bot.strategy.ensemble import ensemble_combine as prod_ensemble, _REGIME_WEIGHTS
from bot.strategy.portfolio_optimizer import normalize_weights as prod_normalize
from bot.risk.risk_manager import enforce_position_limit as prod_position_limit
from bot.risk.circuit_breaker import CircuitBreaker
from bot.execution.order_executor import generate_rebalance_orders as prod_orders

print("Production imports OK ✅")
print(f"\nProduction regime weights:")
for regime_name, weights in _REGIME_WEIGHTS.items():
    formatted = ", ".join(f"{k}={v:.2f}" for k, v in weights.items())
    print(f"  {regime_name:8s}: {formatted}")

# Verify circuit breaker
cb = CircuitBreaker()
for dd_val in [0.01, 0.03, 0.05, 0.08]:
    print(f"  CB({dd_val:.0%} DD) → {cb.evaluate(dd_val)}")

---
## 9 · Weight Evolution Over Time

In [ ]:
# Re-run pipeline to capture weight history
weight_history = []
for day in range(WARMUP, N_DAYS):
    window = prices.iloc[:day + 1]
    reg = detect_regime(window["BTCUSDT"], **CONFIG["regime"])
    mom_w = momentum_signal(window, **CONFIG["momentum"])
    mr_w = mean_reversion_signal(window, **CONFIG["mean_reversion"])
    raw = ensemble_combine(reg, {"momentum": mom_w, "mean_reversion": mr_w})
    norm_w = normalize_weights(raw, CASH_FLOORS[reg])
    risk_w = enforce_position_limit(norm_w, CONFIG["risk"]["max_position"])
    
    row = {"date": dates[day]}
    row.update({a: risk_w.get(a, 0.0) for a in assets})
    row["cash"] = max(0, 1.0 - sum(risk_w.values()))
    weight_history.append(row)

wh_df = pd.DataFrame(weight_history).set_index("date")

fig, ax = plt.subplots(figsize=(14, 6))
cols = assets + ["cash"]
cmap = plt.cm.Set3(np.linspace(0, 1, len(cols)))
ax.stackplot(wh_df.index, *[wh_df[c] for c in cols], labels=cols, colors=cmap, alpha=0.8)
ax.set_ylabel("Weight")
ax.set_xlabel("Date")
ax.set_title("Portfolio Weight Evolution")
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), fontsize=8)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

---
## 10 · Key Takeaways

| Concept | Detail |
|---------|--------|
| **Six Layers** | Data → Signals → Strategy → Portfolio → Risk → Execution |
| **Regime-Adaptive** | Weights shift based on bull/ranging/bear detection |
| **Cash Floor** | 20% (bull), 40% (ranging), 50% (bear) |
| **Circuit Breakers** | L1 (3% DD) → reduce, L2 (5% DD) → halt |
| **State Persistence** | Atomic JSON writes for crash recovery |
| **Scheduler** | APScheduler runs poll (60s) and strategy (300s) jobs |

### What We Built in This Course

| Notebook | Topic | Key Skills |
|----------|-------|-----------|
| NB00 | Course Overview | Architecture, learning path |
| NB01 | Market Data | API calls, OHLCV, SQLite |
| NB02 | Technical Indicators | EMA, RSI, Bollinger Bands |
| NB03 | Risk Metrics | Sharpe, Sortino, VaR, Drawdown |
| NB04 | Momentum Strategy | Multi-lookback scoring, ranking |
| NB05 | Mean Reversion & Pairs | BB signals, cointegration |
| NB06 | Regime Detection | EMA crossover, volatility regime |
| NB07 | Ensemble & Sentiment | Regime weights, F&G, sector rotation |
| NB08 | Portfolio & Risk | Normalization, position limits, circuit breakers |
| NB09 | Backtesting Engine | Walk-forward, metrics, parameter sweep |
| **NB10** | **Full Pipeline** | **End-to-end integration, state persistence** |

---
## 🔬 Exercises

1. **Add Sentiment Overlay:** Generate a synthetic Fear & Greed time series and multiply ensemble weights by a sentiment multiplier (clamped to [0.5, 1.5]).

2. **Transaction Costs:** Add 10bp round-trip cost per rebalance and measure the impact on final portfolio value.

3. **Dynamic Universe:** Start with 4 assets and add 2 more at day 60. How does the pipeline handle new entries?

4. **Telegram Simulation:** Create a mock alerter that prints formatted messages (startup, heartbeat, circuit breaker trigger, daily summary).

5. **Live Data Extension:** Replace synthetic data with real Binance data using `BinanceFetcher` from NB01 and run the full pipeline.

---
## ✅ Knowledge Check

1. What are the six layers of the trading pipeline in order?
2. How does the regime affect the cash floor percentage?
3. What happens when the circuit breaker reaches L2 status?
4. Why does the production bot use atomic writes for state persistence?
5. How often does the production scheduler run the strategy cycle vs the ticker poll?

---
## 🎓 Course Complete!

Congratulations — you've built a **regime-adaptive, multi-strategy crypto trading system** from scratch.

### Next Steps

- Read the production source code in `bot/` to see production-grade patterns
- Run the backtest: `python -m bot.main backtest --symbols BTCUSDT,ETHUSDT`
- Explore `Technicals/06_Strategy_Mathematics_Deep_Dive.md` for the full math
- Check `docs/03_operations_runbook.md` for deployment instructions

Happy trading! 🚀